# Leave-One-Seizure-Out (LOSO) Evaluation Splitting

**Task scope (supervisor's next methodological requirement).** Build a
patient-specific leave-one-seizure-out evaluation SPLITTING framework for
the binary PREICTAL/INTERICTAL labels, for all three horizons (5/15/30
min), with normalization fit strictly inside each fold. **No classifier is
trained here** -- this notebook only constructs, validates, and reports
folds.

**Reused, unmodified inputs:**
- `outputs/labels_binary_preictal.parquet` -- the RAW (unscaled) qEEG
  feature table with binary labels (previous task). This notebook
  deliberately does **not** read `outputs/features_patient_scaled.parquet`
  as a feature source: that file was pre-scaled using a single fixed
  recording-level TRAIN/TEST split, so reusing its `*_scaled` columns here
  would silently leak whatever epochs are TEST in a given LOSO fold into
  the statistics that produced those pre-scaled values. Raw features are
  loaded fresh and re-scaled inside every fold instead.
- `fit_patient_scalers` / `transform_with_patient_scalers` from
  `EEG_patient_normalization.ipynb`, reused (copied verbatim, credited)
  rather than reimplemented.

**New output:** `outputs/loso_fold_manifest.csv` / `.parquet`,
`outputs/loso_ineligible_seizures.csv`, and
`outputs/loso_epoch_fold_context.parquet`. Nothing existing is modified.


## 1. Seizure metadata inspection

Load the labeled table and, per patient, enumerate every annotated seizure
(`seizure_id`, recovered from the `seizure_id` column already computed by
`EEG_binary_preictal_labeling.ipynb` -- every seizure has >=1
seizure-overlapping epoch, so `df["seizure_id"].dropna().unique()` per
patient is the complete, authoritative seizure list; no seizure count is
hardcoded here).


In [1]:

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

print("=" * 60)
print("LOSO EVALUATION SPLITTING -- SETUP")
print("=" * 60)

repo_root = Path.cwd()
output_dir = repo_root / "outputs"
in_path = output_dir / "labels_binary_preictal.parquet"
if not in_path.exists():
    raise FileNotFoundError(f"{in_path} not found -- run EEG_binary_preictal_labeling.ipynb first.")

df = pd.read_parquet(in_path)
print(f"Loaded {in_path.name}: shape={df.shape}")

PATIENT_COL = "patient_id"
HORIZONS = [5, 15, 30]
NON_FEATURE_COLS = {
    "patient_id", "subject_id", "recording_id", "dataset", "epoch_idx",
    "t_start_sec", "t_center_sec", "t_end_sec",
    "y_seizure", "seizure_id", "next_seizure_id", "next_seizure_start_sec",
    "time_to_next_seizure_sec", "previous_seizure_id",
    "previous_seizure_end_sec", "time_since_previous_seizure_sec",
    "binary_label_h5", "binary_label_h15", "binary_label_h30",
    "exclusion_reason_h5", "exclusion_reason_h15", "exclusion_reason_h30",
}
FEATURE_COLS = [c for c in df.columns if c not in NON_FEATURE_COLS]
PATIENTS = sorted(df[PATIENT_COL].unique())
print(f"Feature columns ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print(f"Patients available: {PATIENTS}")


LOSO EVALUATION SPLITTING -- SETUP
Loaded labels_binary_preictal.parquet: shape=(2040, 36)
Feature columns (14): ['std', 'var', 'rms', 'delta', 'theta', 'alpha', 'beta', 'rel_delta', 'rel_theta', 'rel_alpha', 'rel_beta', 'theta_alpha_ratio', 'delta_beta_ratio', 'entropy']
Patients available: ['chb12', 'chb24']


In [2]:

print("Per-patient seizure inventory (from seizure_id -- every seizure has >=1 overlapping epoch):")
seizure_inventory = {}
for pid in PATIENTS:
    pdf = df[df[PATIENT_COL] == pid]
    seizure_ids = sorted(pdf["seizure_id"].dropna().unique())
    recordings_with_seizures = sorted(pdf.loc[pdf["seizure_id"].notna(), "recording_id"].unique())
    seizure_inventory[pid] = seizure_ids
    print(f"\n{pid}: {len(seizure_ids)} annotated seizures across {len(recordings_with_seizures)} recordings")
    print(f"  seizure_ids  : {seizure_ids}")
    print(f"  recordings   : {recordings_with_seizures}")


Per-patient seizure inventory (from seizure_id -- every seizure has >=1 overlapping epoch):

chb12: 19 annotated seizures across 5 recordings
  seizure_ids  : ['chb_s12_r06_sz0', 'chb_s12_r06_sz1', 'chb_s12_r08_sz0', 'chb_s12_r08_sz1', 'chb_s12_r08_sz2', 'chb_s12_r08_sz3', 'chb_s12_r23_sz0', 'chb_s12_r23_sz1', 'chb_s12_r23_sz2', 'chb_s12_r38_sz0', 'chb_s12_r38_sz1', 'chb_s12_r38_sz2', 'chb_s12_r38_sz3', 'chb_s12_r38_sz4', 'chb_s12_r42_sz0', 'chb_s12_r42_sz1', 'chb_s12_r42_sz2', 'chb_s12_r42_sz3', 'chb_s12_r42_sz4']
  recordings   : ['chb_s12_r06', 'chb_s12_r08', 'chb_s12_r23', 'chb_s12_r38', 'chb_s12_r42']

chb24: 16 annotated seizures across 12 recordings
  seizure_ids  : ['chb_s24_r01_sz0', 'chb_s24_r01_sz1', 'chb_s24_r03_sz0', 'chb_s24_r03_sz1', 'chb_s24_r04_sz0', 'chb_s24_r04_sz1', 'chb_s24_r04_sz2', 'chb_s24_r06_sz0', 'chb_s24_r07_sz0', 'chb_s24_r09_sz0', 'chb_s24_r11_sz0', 'chb_s24_r13_sz0', 'chb_s24_r14_sz0', 'chb_s24_r15_sz0', 'chb_s24_r17_sz0', 'chb_s24_r21_sz0']
  recordings 

## 1b. PREICTAL eligibility per seizure per horizon

A seizure is **eligible** for LOSO at horizon H iff it has >=1 PREICTAL
epoch at that horizon (`binary_label_h{H} == "PREICTAL"` and
`next_seizure_id == <this seizure>`). This directly reuses the exclusion
logic already validated in `EEG_binary_preictal_labeling.ipynb` -- no new
labeling rule is introduced here. Ineligible seizures are **reported, never
silently dropped** (their diagnosed exclusion reason is computed from the
`exclusion_reason_h{H}` column already attached to each near-window epoch,
not re-derived from scratch).


In [3]:

def compute_seizure_eligibility(df, patient_id, horizon_minutes, seizure_ids):
    '''
    Returns (eligible_seizure_ids, ineligible_rows) for one patient/horizon.
    eligible = >=1 PREICTAL epoch at this horizon whose next_seizure_id == seizure.
    ineligible_rows: list of {seizure_id, reason} -- diagnosed from
    exclusion_reason_h{H} among the epochs that were candidates for this
    seizure's preictal window (0 <= time_to_next_seizure_sec <= H*60).
    '''
    lab_col = f"binary_label_h{horizon_minutes}"
    reason_col = f"exclusion_reason_h{horizon_minutes}"
    horizon_sec = horizon_minutes * 60.0
    pdf = df[df["patient_id"] == patient_id]

    eligible, ineligible = [], []
    for sid in seizure_ids:
        n_preictal = int(((pdf[lab_col] == "PREICTAL") & (pdf["next_seizure_id"] == sid)).sum())
        if n_preictal > 0:
            eligible.append(sid)
            continue
        candidates = pdf[
            (pdf["next_seizure_id"] == sid)
            & (pdf["time_to_next_seizure_sec"] >= 0)
            & (pdf["time_to_next_seizure_sec"] <= horizon_sec)
        ]
        if len(candidates) == 0:
            reason = "no candidate epochs within horizon (insufficient recording history before onset)"
        else:
            reason_counts = candidates[reason_col].value_counts()
            top_reason = reason_counts.idxmax() if len(reason_counts) else "unknown"
            reason = f"all {len(candidates)} candidate epoch(s) excluded (dominant reason: {top_reason})"
        ineligible.append({"seizure_id": sid, "reason": reason})
    return eligible, ineligible


print("Eligibility per patient per horizon:")
for H in HORIZONS:
    print(f"\n--- H = {H} min ---")
    for pid in PATIENTS:
        eligible, ineligible = compute_seizure_eligibility(df, pid, H, seizure_inventory[pid])
        print(f"  {pid}: {len(eligible)}/{len(seizure_inventory[pid])} seizures eligible")
        print(f"    eligible  : {eligible}")
        if ineligible:
            for row in ineligible:
                print(f"    INELIGIBLE: {row['seizure_id']:<20s} -- {row['reason']}")


Eligibility per patient per horizon:

--- H = 5 min ---
  chb12: 5/19 seizures eligible
    eligible  : ['chb_s12_r06_sz0', 'chb_s12_r08_sz0', 'chb_s12_r23_sz0', 'chb_s12_r38_sz0', 'chb_s12_r42_sz0']
    INELIGIBLE: chb_s12_r06_sz1      -- all 10 candidate epoch(s) excluded (dominant reason: postictal_exclusion)
    INELIGIBLE: chb_s12_r08_sz1      -- all 6 candidate epoch(s) excluded (dominant reason: postictal_exclusion)
    INELIGIBLE: chb_s12_r08_sz2      -- all 10 candidate epoch(s) excluded (dominant reason: postictal_exclusion)
    INELIGIBLE: chb_s12_r08_sz3      -- all 10 candidate epoch(s) excluded (dominant reason: postictal_exclusion)
    INELIGIBLE: chb_s12_r23_sz1      -- all 6 candidate epoch(s) excluded (dominant reason: seizure_overlap)
    INELIGIBLE: chb_s12_r23_sz2      -- all 7 candidate epoch(s) excluded (dominant reason: seizure_overlap)
    INELIGIBLE: chb_s12_r38_sz1      -- all 10 candidate epoch(s) excluded (dominant reason: postictal_exclusion)
    INELIGIBL

## 1c. What this means for LOSO eligibility (the "19 of 35" issue)

The previous task's validation found that only 19 of 35 annotated seizures
(across both patients) contribute any PREICTAL epoch at all, at any
horizon -- the rest are clustered seizures (mostly CHB12) whose preictal
window falls entirely inside the 30-minute postictal exclusion of the
immediately preceding seizure. Re-verified above, per-horizon, from the
actual data (not assumed): the eligible set is identical across H=5/15/30
here (a seizure with zero clean epochs at the tightest 5-minute horizon
also has zero at 15 and 30, because the epochs consumed are the SAME
`postictal_exclusion` epochs regardless of how wide the horizon is; a
seizure's shortage is a property of its position relative to the previous
seizure/recording start, not of the horizon width).

**Effect on LOSO construction:** an ineligible seizure simply CANNOT serve
as a held-out TEST event -- there is no PREICTAL epoch to test detection
against, so "held out this seizure, evaluate on it" is not a meaningful
statement for it. This is not a bug to work around: `seizure_id` /
`next_seizure_id` ARE sufficient to construct an unambiguous held-out fold
for every ELIGIBLE seizure (each PREICTAL/INTERICTAL epoch maps to exactly
one seizure context, see Section 2 below) -- eligibility is a property of
the underlying EEG/annotation data (cluster seizures with no clean
preictal window), not a limitation of the identifier scheme. Ineligible
seizures are excluded from LOSO folds and reported explicitly (Section 7),
never silently dropped from the seizure inventory.


## 2/3. Fold definition: what belongs to TRAIN vs TEST

**PREICTAL epochs** unambiguously belong to the seizure named by their own
`next_seizure_id` -- this is exactly the seizure the epoch is a "run-up"
to, already computed and validated in the labeling task. No new rule is
needed here.

**INTERICTAL epochs are NOT intrinsically tied to one seizure** (that is
the point of "interictal" -- far from any seizure), so a rule must be
chosen for which fold they belong to. Two options were considered:

1. **Whole-recording assignment** (hold out every epoch in the held-out
   seizure's recording). REJECTED: several CHB24 recordings (`r01`, `r03`)
   contain TWO eligible seizures each. Holding out the whole recording for
   seizure A would also remove seizure B's PREICTAL epochs from TRAIN in
   A's fold -- but B must remain in TRAIN when A is held out (every OTHER
   eligible seizure belongs in TRAIN, per the task's own fold definition).
   Recording-level assignment silently violates this whenever a recording
   contains more than one eligible seizure, so it is not scientifically
   defensible here.
2. **Seizure-context assignment via existing temporal-adjacency metadata**
   (adopted): every epoch already carries `next_seizure_id` (the seizure
   it precedes, if any, in its recording) and `previous_seizure_id` (the
   seizure it most recently followed). An INTERICTAL epoch is assigned to
   `next_seizure_id` if defined, else `previous_seizure_id`. Because every
   currently-loaded recording contains at least one seizure, every epoch
   has at least one of these defined (an epoch before a recording's first
   seizure has `next_seizure_id` set; an epoch after its last seizure has
   `previous_seizure_id` set) -- verified programmatically below (no
   silent fallback to an arbitrary default). This gives every labeled
   epoch (PREICTAL or INTERICTAL) exactly one seizure "context", which is
   what makes TRAIN and TEST a strict, disjoint partition per seizure,
   for both PREICTAL and INTERICTAL epochs simultaneously, using metadata
   that was already computed, validated, and audited in
   `EEG_binary_preictal_labeling.ipynb` (no new labeling logic is
   introduced).

For held-out seizure S of patient P: **TEST** = all of P's PREICTAL/
INTERICTAL epochs whose seizure-context equals S. **TRAIN** = all of P's
remaining PREICTAL/INTERICTAL epochs (seizure-context != S). Excluded
epochs (`seizure_overlap`, `postictal_exclusion`, `ambiguous_preictal_gap`,
`seizure_onset_boundary`) never enter either fold. This is chronology- and
recording-structure-respecting because it is derived entirely from
existing recording-relative timing metadata, never from a random epoch
draw.


In [4]:

def assign_fold_context(df, horizon_minutes):
    '''
    Per-epoch seizure 'context' for LOSO fold assignment at this horizon:
      PREICTAL epoch   -> its own next_seizure_id (the seizure it precedes)
      INTERICTAL epoch -> next_seizure_id if defined, else previous_seizure_id
      everything else  -> None (excluded epochs never enter a fold)
    '''
    lab_col = f"binary_label_h{horizon_minutes}"
    fold_context = pd.Series(None, index=df.index, dtype=object)

    is_preictal = (df[lab_col] == "PREICTAL").to_numpy()
    is_interictal = (df[lab_col] == "INTERICTAL").to_numpy()

    assert df.loc[is_preictal, "next_seizure_id"].notna().all(), \
        "A PREICTAL epoch has no next_seizure_id -- cannot assign fold context"
    fold_context.loc[is_preictal] = df.loc[is_preictal, "next_seizure_id"]

    inter_next = df.loc[is_interictal, "next_seizure_id"]
    inter_prev = df.loc[is_interictal, "previous_seizure_id"]
    inter_context = inter_next.where(inter_next.notna(), inter_prev)
    assert inter_context.notna().all(), \
        "An INTERICTAL epoch has neither a next nor a previous seizure in its recording"
    fold_context.loc[is_interictal] = inter_context

    return fold_context


for H in HORIZONS:
    ctx = assign_fold_context(df, H)
    n_assigned = ctx.notna().sum()
    lab_col = f"binary_label_h{H}"
    n_labeled = df[lab_col].notna().sum()
    assert n_assigned == n_labeled, f"H={H}: {n_labeled} labeled epochs but only {n_assigned} got a fold context"
    print(f"[OK] H={H:2d}min: every one of the {n_labeled} PREICTAL/INTERICTAL epochs "
          f"received exactly one seizure fold-context (no epoch left unassigned).")


[OK] H= 5min: every one of the 542 PREICTAL/INTERICTAL epochs received exactly one seizure fold-context (no epoch left unassigned).
[OK] H=15min: every one of the 817 PREICTAL/INTERICTAL epochs received exactly one seizure fold-context (no epoch left unassigned).
[OK] H=30min: every one of the 1104 PREICTAL/INTERICTAL epochs received exactly one seizure fold-context (no epoch left unassigned).


## 4/5. Reusable, horizon-configurable LOSO fold builder, with normalization fit inside each fold

`build_loso_fold(...)` is called once per (patient, horizon, held-out
seizure) -- one function, not three horizon-specific copies. Inside it:

```text
patient data (raw features)
    -> TRAIN = labeled epochs with fold_context != held_out_seizure
    -> TEST  = labeled epochs with fold_context == held_out_seizure
    -> fit_patient_scalers(..., train_mask)      # TRAIN ONLY, this fold
    -> transform_with_patient_scalers(...)       # applied to TRAIN and TEST
```

`fit_patient_scalers` / `transform_with_patient_scalers` are the exact
functions from `EEG_patient_normalization.ipynb` (copied verbatim below,
credited) -- reused, not reimplemented. Since each LOSO fold is already
restricted to one patient, they degenerate to fitting a single scaler per
fold, which is exactly "patient-specific normalization, fit on TRAIN only"
applied consistently inside every fold.


In [5]:

def fit_patient_scalers(df, patient_col, feature_cols, train_mask):
    '''Fit one StandardScaler per patient, using ONLY that patient's TRAIN rows.
    (Verbatim from EEG_patient_normalization.ipynb.)'''
    scalers = {}
    train_mask = np.asarray(train_mask)
    for pid in df.loc[train_mask, patient_col].unique():
        sel = train_mask & (df[patient_col] == pid).to_numpy()
        n = int(sel.sum())
        if n == 0:
            raise ValueError(f"Patient {pid!r} has zero TRAIN rows -- cannot fit a scaler.")
        assert df.loc[sel, patient_col].nunique() == 1
        scaler = StandardScaler()
        scaler.fit(df.loc[sel, feature_cols])
        scalers[pid] = scaler
    return scalers


def transform_with_patient_scalers(df, patient_col, feature_cols, scalers):
    '''Transform every row using ITS OWN patient's fitted scaler.
    (Verbatim from EEG_patient_normalization.ipynb.)'''
    out = df.copy()
    scaled = np.full((len(df), len(feature_cols)), np.nan)
    covered = np.zeros(len(df), dtype=bool)
    for pid, scaler in scalers.items():
        sel = (df[patient_col] == pid).to_numpy()
        if sel.any():
            scaled[sel, :] = scaler.transform(df.loc[sel, feature_cols])
            covered |= sel
    if not covered.all():
        missing_pids = sorted(df.loc[~covered, patient_col].unique())
        raise ValueError(f"No fitted scaler available for patients: {missing_pids}")
    scaled_cols = [f"{c}_scaled" for c in feature_cols]
    out[scaled_cols] = scaled
    return out, scaled_cols


def build_loso_fold(patient_df, horizon_minutes, held_out_seizure_id, fold_context, feature_cols, patient_col="patient_id"):
    '''Build one LOSO fold for ONE patient, ONE horizon, ONE held-out seizure.'''
    lab_col = f"binary_label_h{horizon_minutes}"
    labeled_mask = patient_df[lab_col].notna().to_numpy()
    ctx = fold_context.loc[patient_df.index].to_numpy()

    test_mask = labeled_mask & (ctx == held_out_seizure_id)
    train_mask = labeled_mask & (ctx != held_out_seizure_id)

    scalers = fit_patient_scalers(patient_df, patient_col, feature_cols, train_mask)
    scaled_df, scaled_cols = transform_with_patient_scalers(patient_df, patient_col, feature_cols, scalers)

    pid = patient_df[patient_col].iloc[0]
    return {
        "patient_id": pid,
        "horizon_minutes": horizon_minutes,
        "held_out_seizure_id": held_out_seizure_id,
        "lab_col": lab_col,
        "train_idx": patient_df.index[train_mask],
        "test_idx": patient_df.index[test_mask],
        "scaler": scalers[pid],
        "scaled_df": scaled_df,
        "scaled_cols": scaled_cols,
    }


print("[OK] build_loso_fold() defined -- one reusable function for all patients/horizons/seizures.")


[OK] build_loso_fold() defined -- one reusable function for all patients/horizons/seizures.


## 6. Strict leakage checks (run for every single fold, not a sample)

In [6]:

def validate_loso_fold(fold, patient_df, feature_cols):
    train_idx, test_idx = fold["train_idx"], fold["test_idx"]
    lab_col = fold["lab_col"]
    held_out = fold["held_out_seizure_id"]

    # 1. train and test row indices are disjoint
    assert len(set(train_idx) & set(test_idx)) == 0, "TRAIN/TEST indices overlap"

    # 2. held-out seizure's PREICTAL epochs never occur in TRAIN
    train_next = patient_df.loc[train_idx, "next_seizure_id"]
    train_preictal = patient_df.loc[train_idx, lab_col] == "PREICTAL"
    assert not (train_preictal & (train_next == held_out)).any(), \
        f"Held-out seizure {held_out} has a PREICTAL epoch leaking into TRAIN"

    # 3/4. scaler.fit() received TRAIN rows only; no TEST row contributed
    scaler = fold["scaler"]
    assert int(scaler.n_samples_seen_) == len(train_idx), \
        f"scaler.n_samples_seen_={scaler.n_samples_seen_} != len(train_idx)={len(train_idx)}"
    manual_mean = patient_df.loc[train_idx, feature_cols].to_numpy().mean(axis=0)
    assert np.allclose(manual_mean, scaler.mean_), "scaler mean does not match TRAIN-only recomputation"

    # 5. only one patient exists in this fold
    combined = pd.concat([patient_df.loc[train_idx, "patient_id"], patient_df.loc[test_idx, "patient_id"]])
    assert combined.nunique() == 1, "More than one patient present in a single LOSO fold"

    # 6. feature columns/order identical
    assert list(scaler.feature_names_in_) == list(feature_cols), "Feature column order mismatch at fit time"

    # 7. excluded epochs never used for modeling
    assert patient_df.loc[train_idx, lab_col].isin(["PREICTAL", "INTERICTAL"]).all()
    assert patient_df.loc[test_idx, lab_col].isin(["PREICTAL", "INTERICTAL"]).all()

    # 8. seizure-overlapping epochs never used
    assert (patient_df.loc[train_idx, "y_seizure"] == 0).all(), "A seizure-overlapping epoch is in TRAIN"
    assert (patient_df.loc[test_idx, "y_seizure"] == 0).all(), "A seizure-overlapping epoch is in TEST"

    # 9. chronological metadata preserved
    all_idx = train_idx.union(test_idx)
    for col in ["recording_id", "epoch_idx", "t_start_sec", "t_end_sec"]:
        assert patient_df.loc[all_idx, col].notna().all(), f"Metadata column {col} has nulls in this fold"

    return True


print("[OK] validate_loso_fold() defined (9 checks).")


[OK] validate_loso_fold() defined (9 checks).


## 7/8. Build, validate, and report every fold for every patient and horizon

In [7]:

manifest_rows = []
ineligible_rows = []
n_folds_built = 0
n_folds_validated = 0

for H in HORIZONS:
    fold_context = assign_fold_context(df, H)
    lab_col = f"binary_label_h{H}"

    for pid in PATIENTS:
        patient_df = df[df[PATIENT_COL] == pid].copy()
        eligible, ineligible = compute_seizure_eligibility(df, pid, H, seizure_inventory[pid])

        for row in ineligible:
            ineligible_rows.append({"patient_id": pid, "horizon_minutes": H, **row})

        for fold_id, sid in enumerate(eligible, start=1):
            fold = build_loso_fold(patient_df, H, sid, fold_context, FEATURE_COLS)
            n_folds_built += 1
            validate_loso_fold(fold, patient_df, FEATURE_COLS)
            n_folds_validated += 1

            train_labels = patient_df.loc[fold["train_idx"], lab_col]
            test_labels = patient_df.loc[fold["test_idx"], lab_col]
            manifest_rows.append({
                "patient_id": pid,
                "horizon_minutes": H,
                "fold_id": fold_id,
                "held_out_seizure_id": sid,
                "train_preictal_count": int((train_labels == "PREICTAL").sum()),
                "train_interictal_count": int((train_labels == "INTERICTAL").sum()),
                "test_preictal_count": int((test_labels == "PREICTAL").sum()),
                "test_interictal_count": int((test_labels == "INTERICTAL").sum()),
                "train_total": int(len(fold["train_idx"])),
                "test_total": int(len(fold["test_idx"])),
            })

manifest_df = pd.DataFrame(manifest_rows)
ineligible_df = pd.DataFrame(ineligible_rows)

print(f"Folds built  : {n_folds_built}")
print(f"Folds validated (all 9 leakage checks passed): {n_folds_validated}")
assert n_folds_built == n_folds_validated
print("\n[ALL LOSO FOLD LEAKAGE CHECKS PASSED]")


Folds built  : 57
Folds validated (all 9 leakage checks passed): 57

[ALL LOSO FOLD LEAKAGE CHECKS PASSED]


In [8]:

print("=" * 60)
print("VALIDATION REPORT")
print("=" * 60)

print(f"\nPatients: {PATIENTS}")

for H in HORIZONS:
    hm = manifest_df[manifest_df["horizon_minutes"] == H]
    im = ineligible_df[ineligible_df["horizon_minutes"] == H]
    print(f"\n--- H = {H} minutes ---")
    print(f"Total LOSO folds       : {len(hm)}")
    for pid in PATIENTS:
        n_eligible = (hm["patient_id"] == pid).sum()
        n_ineligible = (im["patient_id"] == pid).sum()
        print(f"  {pid}: {n_eligible} eligible seizures (folds), {n_ineligible} ineligible")
    print(f"\nPer-fold class counts:")
    print(hm[["patient_id", "fold_id", "held_out_seizure_id",
              "train_preictal_count", "train_interictal_count",
              "test_preictal_count", "test_interictal_count",
              "train_total", "test_total"]].to_string(index=False))

    low_test = hm[hm["test_preictal_count"] <= 2]
    if len(low_test):
        print(f"\n  [LOW SAMPLE WARNING] {len(low_test)} fold(s) with <=2 TEST PREICTAL epochs "
              f"(evaluation on these folds will be extremely noisy):")
        print(low_test[["patient_id", "held_out_seizure_id", "test_preictal_count"]].to_string(index=False))

    zero_test_inter = hm[hm["test_interictal_count"] == 0]
    if len(zero_test_inter):
        print(f"\n  [ZERO-INTERICTAL WARNING] {len(zero_test_inter)} fold(s) with 0 TEST INTERICTAL "
              f"epochs -- for these held-out seizures, no INTERICTAL epoch's fold-context (its own "
              f"next/previous seizure) happens to equal this seizure: either the recording is so "
              f"seizure-dense that none of its epochs are ever >=30min from EVERY seizure in it "
              f"(CHB24 r01/r03/r04), or the recording's few INTERICTAL epochs are contextually tied "
              f"to one of its OTHER (later, ineligible) seizures rather than to this one (CHB12's "
              f"eligible seizure is always each recording's FIRST -- its own interictal room mostly "
              f"falls between LATER seizures instead). TEST for these folds is PREICTAL-only, so "
              f"specificity/false-positive-rate cannot be evaluated on these folds in isolation:")
        print(zero_test_inter[["patient_id", "held_out_seizure_id"]].to_string(index=False))


VALIDATION REPORT

Patients: ['chb12', 'chb24']

--- H = 5 minutes ---
Total LOSO folds       : 19
  chb12: 5 eligible seizures (folds), 14 ineligible
  chb24: 14 eligible seizures (folds), 2 ineligible

Per-fold class counts:
patient_id  fold_id held_out_seizure_id  train_preictal_count  train_interictal_count  test_preictal_count  test_interictal_count  train_total  test_total
     chb12        1     chb_s12_r06_sz0                    38                      37                   10                      0           75          10
     chb12        2     chb_s12_r08_sz0                    38                      37                   10                      0           75          10
     chb12        3     chb_s12_r23_sz0                    40                      37                    8                      0           77           8
     chb12        4     chb_s12_r38_sz0                    38                      37                   10                      0           75          1

## Save outputs (new files, nothing existing overwritten)

In [9]:

manifest_parquet = output_dir / "loso_fold_manifest.parquet"
manifest_csv = output_dir / "loso_fold_manifest.csv"
manifest_df.to_parquet(manifest_parquet, index=False)
manifest_df.to_csv(manifest_csv, index=False)

ineligible_csv = output_dir / "loso_ineligible_seizures.csv"
ineligible_df.to_csv(ineligible_csv, index=False)

# Bonus reusable artifact: per-epoch fold context for every horizon, in long
# format -- lets a future modeling notebook reconstruct any fold's TRAIN/TEST
# row set without recomputing assign_fold_context().
context_rows = []
for H in HORIZONS:
    ctx = assign_fold_context(df, H)
    lab_col = f"binary_label_h{H}"
    labeled = df[lab_col].notna()
    part = df.loc[labeled, ["patient_id", "recording_id", "epoch_idx",
                             "t_start_sec", "t_end_sec"]].copy()
    part["horizon_minutes"] = H
    part["binary_label"] = df.loc[labeled, lab_col]
    part["fold_context_seizure_id"] = ctx.loc[labeled]
    context_rows.append(part)
epoch_context_df = pd.concat(context_rows, ignore_index=True)
epoch_context_path = output_dir / "loso_epoch_fold_context.parquet"
epoch_context_df.to_parquet(epoch_context_path, index=False)

print(f"[OK] Saved manifest ({manifest_df.shape[0]} rows)      -> {manifest_parquet}")
print(f"[OK] Saved manifest (csv)                    -> {manifest_csv}")
print(f"[OK] Saved ineligible seizures ({ineligible_df.shape[0]} rows) -> {ineligible_csv}")
print(f"[OK] Saved per-epoch fold context ({epoch_context_df.shape[0]} rows) -> {epoch_context_path}")

import os
for f in ["labels_binary_preictal.parquet", "features_patient_scaled.parquet",
          "features_chb24_forecasting.parquet", "features_chb12_forecasting.parquet"]:
    assert os.path.exists(output_dir / f)
print("\n[OK] All previously existing outputs remain untouched.")


[OK] Saved manifest (57 rows)      -> C:\Users\angela\Downloads\EEG_timeseries-main\EEG_timeseries-main\outputs\loso_fold_manifest.parquet
[OK] Saved manifest (csv)                    -> C:\Users\angela\Downloads\EEG_timeseries-main\EEG_timeseries-main\outputs\loso_fold_manifest.csv
[OK] Saved ineligible seizures (48 rows) -> C:\Users\angela\Downloads\EEG_timeseries-main\EEG_timeseries-main\outputs\loso_ineligible_seizures.csv
[OK] Saved per-epoch fold context (2463 rows) -> C:\Users\angela\Downloads\EEG_timeseries-main\EEG_timeseries-main\outputs\loso_epoch_fold_context.parquet

[OK] All previously existing outputs remain untouched.


## Summary

- **Patients**: `chb12`, `chb24` (the only two currently available), never
  mixed inside any fold (leakage check 5, passed for every fold).
- **No classifier, no new metric, no channel change, no new patient** was
  introduced.
- **Old recording-level split preserved**: `EEG_TCN_state_deviation_forecasting.ipynb`
  and its recording-level TRAIN/VAL/TEST assignment, and
  `EEG_patient_normalization.ipynb`'s recording-level demonstration fold,
  are both untouched -- kept for historical comparison. LOSO is an
  additional, separate evaluation path, not a replacement.
- **Still not implemented** (explicitly out of scope per the task): any
  Logistic Regression / Random Forest / TCN training or evaluation on
  these folds, new evaluation metrics, channel representation changes, and
  expansion beyond CHB24/CHB12.
